# Analysis of Atlas + Mixl1 chimaera

### Import packages

In [11]:
import sys
import numpy as np
import pandas as pd
import scanpy as sc
#import scvelo as scv
#import cellrank as cr
#import bbknn
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import random
random.seed(123)

In [12]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
#scv.set_figure_params()

scanpy==1.7.1 anndata==0.7.5 umap==0.4.6 numpy==1.20.1 scipy==1.6.1 pandas==1.2.2 scikit-learn==0.24.1 statsmodels==0.12.2


### I/O

In [13]:
out_folder = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/data/"
Data = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/wildtype_injection/chimera-wt/data/wt_chim_atlas.h5"
PCs = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/mb2338/annabelle/12_march/corrected_pc_complete.csv" 
meta = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/atlas/atlas/meta.tab"
meta_wt = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/wildtype_injection/chimera-wt/data/big_meta_mapping+haem+meso+gut.tab"
gene_conversion = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/gene_id_name_conversion.csv"

### Load data

In [14]:
adata = sc.read(Data)

In [15]:
adata

AnnData object with n_obs × n_vars = 129793 × 29453
    obs: 'barcode', 'sample', 'stage', 'sequencing.batch', 'stripped', 'doublet', 'stage.mapped', 'celltype.mapped', 'closest.cell', 'tdTom', 'sizeFactor'
    obsm: 'X_pca'
    layers: 'logcounts'

In [16]:
#meta = pd.read_csv(meta, '\t')

In [17]:
meta_wt = pd.read_csv(meta_wt, '\t')

In [18]:
meta_wt.set_index('cell', inplace=True)

In [19]:
meta_wt.columns

Index(['X.1', 'X', 'barcode', 'batch', 'celltype', 'celltype.mapped',
       'closest.cell', 'cluster', 'cluster.stage', 'cluster.sub',
       'cluster.theiler', 'colour', 'doub.density', 'doublet', 'endo_gephiX',
       'endo_gephiY', 'endo_gutCluster', 'endo_gutDPT', 'endo_gutX',
       'endo_gutY', 'endo_trajectoryDPT', 'endo_trajectoryName', 'haem_gephiX',
       'haem_gephiY', 'haem_subclust', 'sample', 'sequencing.batch',
       'sizeFactor', 'stage', 'stage.mapped', 'stripped', 'tdTom', 'theiler',
       'umapX', 'umapY', 'origin', 'haem_subclust.mapped', 'celltype_origin',
       'meso_subcluster', 'meso.mapped', 'gut_subclust.mapped'],
      dtype='object')

In [20]:
#meta['cell'] = meta['cell'].apply(lambda x: f"atlas_{x}") #add atlas to cell names

In [21]:
adata.obs

,barcode,sample,stage,sequencing.batch,stripped,doublet,stage.mapped,celltype.mapped,closest.cell,tdTom,sizeFactor
cell,,,,,,,,,,,
atlas_cell_1,AAAGGCCTCCACAA,atlas_1,E6.5,1,False,False,nan,nan,nan,nan,0.466785
atlas_cell_2,AACAAACTCGCCTT,atlas_1,E6.5,1,False,False,nan,nan,nan,nan,1.029076
atlas_cell_5,AACAGAGAATCAGC,atlas_1,E6.5,1,False,False,nan,nan,nan,nan,0.907052
atlas_cell_6,AACATATGAATCGC,atlas_1,E6.5,1,False,False,nan,nan,nan,nan,1.129780
atlas_cell_8,AACCGATGGCTTCC,atlas_1,E6.5,1,False,False,nan,nan,nan,nan,1.206612
...,...,...,...,...,...,...,...,...,...,...,...
wt_cell_13477,TTTGGTTGTGATGCCC,mixl_6,E8.5,0,False,False,E7.75,Rostral neurectoderm,atlas_cell_8750,neg,0.422911
wt_cell_13478,TTTGGTTTCGCTTAGA,mixl_6,E8.5,0,False,False,E8.5,Erythroid3,atlas_cell_97649,neg,1.336606
wt_cell_13479,TTTGTCAAGACAATAC,mixl_6,E8.5,0,False,False,E8.5,Erythroid3,atlas_cell_38655,neg,2.407617


In [22]:
meta_wt

,X.1,X,barcode,batch,celltype,celltype.mapped,closest.cell,cluster,cluster.stage,cluster.sub,...,tdTom,theiler,umapX,umapY,origin,haem_subclust.mapped,celltype_origin,meso_subcluster,meso.mapped,gut_subclust.mapped
cell,,,,,,,,,,,,,,,,,,,,,
atlas_cell_1,1,atlas_cell_1-0,AAAGGCCTCCACAA,0,Epiblast,NaN,NaN,2.0,2.0,4.0,...,NaN,TS9,-10.227546,-2.881687,atlas,NaN,NaN,NaN,NaN,NaN
atlas_cell_2,2,atlas_cell_2-0,AACAAACTCGCCTT,0,Primitive Streak,NaN,NaN,12.0,1.0,1.0,...,NaN,TS9,-6.625458,0.108961,atlas,NaN,NaN,NaN,NaN,NaN
atlas_cell_5,3,atlas_cell_5-0,AACAGAGAATCAGC,0,ExE ectoderm,NaN,NaN,3.0,4.0,7.0,...,NaN,TS9,10.061009,-0.029313,atlas,NaN,NaN,NaN,NaN,NaN
atlas_cell_6,4,atlas_cell_6-0,AACATATGAATCGC,0,Epiblast,NaN,NaN,1.0,3.0,1.0,...,NaN,TS9,-10.454418,-0.269452,atlas,NaN,NaN,NaN,NaN,NaN
atlas_cell_8,5,atlas_cell_8-0,AACCGATGGCTTCC,0,Epiblast,NaN,NaN,2.0,2.0,1.0,...,NaN,TS9,-11.047206,-2.205269,atlas,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wt_cell_13477,129789,wt_cell_13477-1,TTTGGTTGTGATGCCC,1,Rostral neurectoderm,Rostral neurectoderm,atlas_cell_8750,NaN,NaN,NaN,...,neg,NaN,NaN,NaN,wt,NaN,NaN,NaN,NaN,NaN
wt_cell_13478,129790,wt_cell_13478-1,TTTGGTTTCGCTTAGA,1,Erythroid3,Erythroid3,atlas_cell_97649,NaN,NaN,NaN,...,neg,NaN,NaN,NaN,wt,Ery4,NaN,NaN,NaN,NaN
wt_cell_13479,129791,wt_cell_13479-1,TTTGTCAAGACAATAC,1,Erythroid3,Erythroid3,atlas_cell_38655,NaN,NaN,NaN,...,neg,NaN,NaN,NaN,wt,Ery3,NaN,NaN,NaN,NaN


In [23]:
#meta.index = meta.cell

In [24]:
#adata_atlas = adata[adata.obs_names.str.startswith('atlas'),:].copy()


In [25]:
#adata_wt = adata[adata.obs_names.str.startswith('wt'),:].copy()

In [26]:
#adata_atlas.shape

In [ ]:
#adata_wt.shape

In [9]:
adata.shape

(129793, 29453)

In [10]:
meta_wt.shape

(129793, 41)

In [ ]:
#meta_new = meta.loc[adata_atlas.obs_names,:].copy()

In [ ]:
meta_new = meta.loc[adata_atlas.obs_names,:].copy()

In [ ]:
meta_new.shape

In [ ]:
adata_atlas.obs = meta_new

In [ ]:
adata_atlas.obs

In [ ]:
adata_atlas.shape

In [ ]:
adata_2 = adata_atlas.concatenate(adata_wt)

In [ ]:
adata.obs = adata_2.obs

In [ ]:
adata.shape

In [ ]:
adata_2.write_loom("/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/wildtype_injection/chimera-wt/wt_atlas_complete.h5")

In [ ]:
adata_test = sc.read_loom("/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/wildtype_injection/chimera-wt/data/atlas_only.h5")

In [ ]:
# update metadata
adata_2.obs['celltype'] = np.where(adata_2.obs['tdTom'] != 'nan', adata_2.obs['celltype.mapped'], adata_2.obs['celltype'])

In [ ]:
sc.pp.neighbors(adata_2, n_neighbors=10, n_pcs=50)

In [ ]:
sc.tl.umap(adata_2)

In [ ]:
adata_2.uns['celltype_colors'] = ['#532C8A', '#c19f70', '#f9decf', '#c9a997', '#B51D8D', '#9e6762', '#3F84AA', '#354E23', '#F397C0', '#ff891c', '#635547', '#C72228', '#f79083', '#EF4E22', '#989898', '#7F6874', '#8870ad', '#647a4f', '#EF5A9D', '#FBBE92', '#139992', '#cc7818', '#DFCDE4', '#C594BF', '#C3C388', '#8EC792', '#0F4A9C', '#8DB5CE', '#1A1A1A', '#FACB12', '#C9EBFB', '#DABE99', '#65A83E', '#005579', '#CDE088', '#f7f79e', '#F6BFCB', ]

In [ ]:
sc.pl.umap(adata_2, color='celltype')

In [ ]:
adata = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/wildtype_injection/chimera-wt/data/wt_chim_atlas_analysis.h5"

In [ ]:
adata_2.write_loom(Results)

In [ ]:
adata = sc.read(Results)

In [ ]:
adata_2.uns['stage_colors'] = ["#D53E4F", "#F46D43",  "#FDAE61", "#FFFFBF", "#FEE08B",  "#E6F598", "#ABDDA4", "#3288BD",  "#66C2A5",  "#A9A9A9"] 

In [ ]:
sc.pl.umap(adata_2, color='stage')

### Save UMAP

In [ ]:
adata = sc.read(Results)

In [ ]:
adata

In [ ]:
sc.settings.figdir = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/aamw4/bt392/mouse/Mixl1_KO/plots/'
sc.pl.umap(adata_2, color='celltype', save='wt_celltypes.png')

In [ ]:
sc.pl.umap(adata_2, color='stage', save='wt_stage.png')

In [ ]:
adata_2.uns['tdTom_colors'] = ['#808080', '#000000','#FF0000'] 

In [ ]:
sc.pl.umap(adata_2, color='tdTom', save='wt_tdtom.png')

In [ ]:
adata_2.write_loom(Results, write_obsm_varm=True)

### Caculate adjecency scores for chimaera cells

In [ ]:
# whole object

In [ ]:
adata = sc.read_loom(Results)

In [ ]:
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')

In [ ]:
X_pca = adata.obsm['X_pca']
pca_atlas = sc.AnnData(X_pca[adata.obs['origin'] == 'atlas']).X
pca_chimaera = sc.AnnData(X_pca[adata.obs['origin'] == 'chimaera']).X

k=15

cell_distances_ct = cdist(pca_chimaera, pca_atlas, metric='euclidean')

Adj = np.zeros(cell_distances_ct.shape, dtype=float)
indices = np.zeros((cell_distances_ct.shape[0], k), dtype=np.int_)

for irow, row in enumerate(cell_distances_ct):
    idcs = np.argpartition(row, k)[:k]
    indices[irow] = idcs

for irow, row in enumerate(indices):
    Adj[irow, row] = 1

adjacency_score = np.sum(Adj, axis=0)

atlas = adata.obs[adata.obs['origin'] == 'atlas']
n_ct = len(atlas.index)

adjacency_df = pd.DataFrame({'cell' : atlas.index, 'adjecency_score_chim' : adjacency_score/n_ct*1000})
#adata.obs['adjacency_score_chim'] = adjacency_score/n_ct*1000

In [ ]:
adjacency_df.to_csv(out_folder + 'adjacency.csv')

In [ ]:
adata = sc.read(Results)
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')
atlas = adata[adata.obs['origin'] == 'atlas']

adjacency_df = pd.read_csv(out_folder + 'adjacency.csv', index_col=0)

In [ ]:
obs_adjecency = adata.obs.merge(adjacency_df, on='cell')
obs_adjecency.set_index(obs_adjecency['cell'], drop=True)
atlas.obs = obs_adjecency
atlas.obs_names = atlas.obs['cell']

In [ ]:
atlas.obs

In [ ]:
atlas

In [ ]:
sc.pl.umap(atlas, color='adjecency_score_chim', size=100)

In [ ]:
sc.pl.draw_graph(atlas, color='adjacency_score_chim', size=100, layout='gephi',
                 color_map=color_map)

In [ ]:
# distance chim from atlas instead of other way around
adata = sc.read(Results)
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')
X_pca = adata.obsm['X_pca']
pca_atlas = sc.AnnData(X_pca[adata.obs['origin'] == 'atlas']).X
pca_chimaera = sc.AnnData(X_pca[adata.obs['origin'] == 'chimaera']).X

k=15

print('calculating distances')
cell_distances_ct = cdist(pca_atlas,pca_chimaera, metric='euclidean')
print('finished distances')

Adj = np.zeros(cell_distances_ct.shape, dtype=float)
indices = np.zeros((cell_distances_ct.shape[0], k), dtype=np.int_)

for irow, row in enumerate(cell_distances_ct):
    idcs = np.argpartition(row, k)[:k]
    indices[irow] = idcs

for irow, row in enumerate(indices):
    Adj[irow, row] = 1

adjacency_score = np.sum(Adj, axis=0)

print('finished rest')

chim = adata[adata.obs['origin'] == 'chimaera']
chim.obs['adjacency_score_chim'] = adjacency_score


In [ ]:
chim.obs.to_csv(out_folder + 'chimaera_distances_complete.csv')

In [ ]:
sc.pl.umap(chim, color='adjacency_score_chim', size=100)

In [ ]:
### per celltype  
k=15


X_pca = adata.obsm['X_pca']
pca_atlas = sc.AnnData(X_pca[adata.obs['origin'] == 'atlas']).X
adata_atlas = adata[adata.obs['origin'] == 'atlas']

chim = adata[adata.obs['origin'] == 'chimaera']

cell_type_list = chim.obs['celltype'].unique()

for ct in cell_type_list:
    print("Calculating neighbour scores for " + ct)
    chim_ct_adata = chim[chim.obs['celltype'] == ct]
    chim_ct_pcs = chim_ct_adata.obsm['X_pca']
    
    cell_distances_ct = cdist(chim_ct_pcs, pca_atlas, metric='euclidean')
    
    Adj = np.zeros(cell_distances_ct.shape, dtype=float)
    indices = np.zeros((cell_distances_ct.shape[0], k), dtype=np.int_)

    for irow, row in enumerate(cell_distances_ct):
        idcs = np.argpartition(row, k)[:k]
        indices[irow] = idcs

    for irow, row in enumerate(indices):
        Adj[irow, row] = 1
        
    adjacency_score = np.sum(Adj, axis=0)
    
    n_ct = len(chim_ct_adata.obs.index)
    adata_atlas.obs['adjacency_score_' + ct] = adjacency_score/n_ct*1000

In [ ]:
sc.settings.figdir = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/plots/'
cell_type_list = sorted(cell_type_list)
sc.pl.umap(adata_atlas, color=['adjacency_score_' + ct for ct in cell_type_list], size=100, save='distances.png')
adata_atlas.obs.to_csv(out_folder + 'atlas_chim_distances.csv')

#### test distances

In [ ]:
#### No normalisation for number of cells per celltype

### per celltype  
k=15
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')


X_pca = adata.obsm['X_pca']
pca_atlas = sc.AnnData(X_pca[adata.obs['origin'] == 'atlas']).X
adata_atlas = adata[adata.obs['origin'] == 'atlas']

chim = adata[adata.obs['origin'] == 'chimaera']

cell_type_list = chim.obs['celltype'].unique()

for ct in cell_type_list:
    print("Calculating neighbour scores for " + ct)
    chim_ct_adata = chim[chim.obs['celltype'] == ct]
    chim_ct_pcs = chim_ct_adata.obsm['X_pca']
    
    cell_distances_ct = cdist(chim_ct_pcs, pca_atlas, metric='euclidean')
    
    Adj = np.zeros(cell_distances_ct.shape, dtype=float)
    indices = np.zeros((cell_distances_ct.shape[0], k), dtype=np.int_)

    for irow, row in enumerate(cell_distances_ct):
        idcs = np.argpartition(row, k)[:k]
        indices[irow] = idcs

    for irow, row in enumerate(indices):
        Adj[irow, row] = 1
        
    adjacency_score = np.sum(Adj, axis=0)
    
    n_ct = len(chim_ct_adata.obs.index)
    adata_atlas.obs['adjacency_score_' + ct] = adjacency_score

In [ ]:
adata_atlas.obs.to_csv(out_folder + 'atlas_chim_distances_no_normalisation.csv')

In [ ]:
#### distance from atlas instead of from chim

### per celltype  
k=15
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')


X_pca = adata.obsm['X_pca']
pca_atlas = sc.AnnData(X_pca[adata.obs['origin'] == 'atlas']).X
adata_atlas = adata[adata.obs['origin'] == 'atlas']

chim = adata[adata.obs['origin'] == 'chimaera']
pca_chim = sc.AnnData(X_pca[adata.obs['origin'] == 'chimaera']).X

cell_type_list = chim.obs['celltype'].unique()

for ct in cell_type_list:
    print("Calculating neighbour scores for " + ct)
    atlas_ct = adata_atlas[adata_atlas.obs['celltype'] == ct]
    atlas_ct_pcs = atlas_ct.obsm['X_pca']
    
    cell_distances_ct = cdist(atlas_ct_pcs, pca_chim, metric='euclidean')
    
    Adj = np.zeros(cell_distances_ct.shape, dtype=float)
    indices = np.zeros((cell_distances_ct.shape[0], k), dtype=np.int_)

    for irow, row in enumerate(cell_distances_ct):
        idcs = np.argpartition(row, k)[:k]
        indices[irow] = idcs

    for irow, row in enumerate(indices):
        Adj[irow, row] = 1
        
    adjacency_score = np.sum(Adj, axis=0)
    
    n_ct = len(chim_ct_adata.obs.index)
    chim.obs['adjacency_score_' + ct] = adjacency_score

In [ ]:
chim.obs.to_csv(out_folder + 'chimaera_distances_no_normalisation.csv')

In [ ]:
sc.settings.figdir = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/plots/'
cell_type_list = sorted(cell_type_list)
sc.pl.umap(chim, color=['adjacency_score_' + ct for ct in cell_type_list], size=100, save='_chimaera_distances.png')

In [ ]:
ct= 'Erythroid3'
print("Calculating neighbour scores for " + ct)
atlas_ct = adata_atlas[adata_atlas.obs['celltype'] == ct]
atlas_ct_pcs = atlas_ct.obsm['X_pca']

cell_distances_ct = cdist(atlas_ct_pcs, pca_chim, metric='euclidean')

In [ ]:
pd.DataFrame(cell_distances_ct)

In [ ]:
pd.DataFrame(pca_chim)

In [ ]:
adata = sc.read(Results)

In [ ]:
umap = pd.DataFrame(adata.obsm['X_umap'])
umap.index = adata.obs.index
umap.columns = ['umap_1', 'umap_2']
umap.to_csv(out_folder + 'big_umap.csv')

In [ ]:
plot = ['celltype.mapped','tdTom']
sc.pl.umap(chim, color = plot, ncols = 2)
sc.pl.umap(chim, color = combined.loc[0], ncols = 2)

In [ ]:
plot = ['celltype.mapped','tdTom']
sc.pl.umap(chim, color = plot, ncols = 2)
sc.pl.umap(chim, color = combined.loc[0], ncols = 2)

In [ ]:
sc.pl.umap(chim, color = 'Xist', ncols = 2)

In [ ]:
sc.pl.umap(chim, color = 'Mixl1', ncols = 2)

In [ ]:
chim_ct = chim[chim.obs['celltype'] == 'Erythroid3']

In [ ]:
sc.pl.umap(adata, color=['Hba-x'])

In [ ]:
adata_old = sc.read(Data)

In [ ]:
adata_old.obsm['X_umap'] = adata.obsm['X_umap']

In [ ]:
sc.pl.umap(adata_old, color=['ENSMUSG00000055609'])

In [ ]:
sc.pp.filter_genes(adata, min_cells=20)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=['Hba-x'])

In [ ]:
sc.pl.umap(adata, color=['Nanog'])

### PAGA - Does not work that well

In [ ]:
adata = sc.read(Results)

In [ ]:
adata.obs['origin'] = np.where(adata.obs['tdTom'] != 'nan', 'chimaera', 'atlas')
adata = adata[adata.obs['origin'] == 'chimaera']

In [ ]:
adata

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)

In [ ]:
sc.tl.paga(adata, groups='celltype')

In [ ]:
sc.pl.paga(adata, color='tdTom', threshold = 0.25, edge_width_scale = 0.2, fontsize = 5)

In [ ]:
sc.pl.violin(adata, 'Hba-x', groupby='celltype')

In [ ]:
adata = sc.read(Results)

In [ ]:
sc.settings.figdir = '/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/plots/'
sc.pl.umap(adata, color='celltype', save='_celltypes.pdf')